# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbottabad123/flyrank-ml-track/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

from huggingface_hub import hf_hub_download
import duckdb
import pandas as pd
import os

con = duckdb.connect()

dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=hf_token
)

fact_march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)


dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
signal1 = con.execute(f"""
    SELECT
        dc.content_hash_id,
        dc.client_hash_id,
        dc.content_updated_date,
        DATE_DIFF('day', dc.content_updated_date, DATE '2026-03-01') as days_since_update,
        AVG(f.gsc_avg_position) as avg_position
    FROM '{dim_content_path}' dc
    JOIN '{fact_march_path}' f
        ON dc.content_hash_id = f.content_hash_id AND dc.client_hash_id = f.client_hash_id
    WHERE dc.is_deleted IS FALSE AND f.gsc_data_available IS TRUE
    GROUP BY dc.content_hash_id, dc.client_hash_id, dc.content_updated_date
""").df()

signal1['staleness_bucket'] = pd.cut(
    signal1['days_since_update'],
    bins=[-1, 30, 90, 100000],
    labels=['fresh_0_30d', 'aging_30_90d', 'stale_90d_plus']
)

bucket_table_1 = signal1.groupby('staleness_bucket').agg(
    n=('content_hash_id', 'count'),
    avg_position=('avg_position', 'mean')
).reset_index()

print(bucket_table_1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  staleness_bucket      n  avg_position
0      fresh_0_30d  25347     14.835140
1     aging_30_90d    437     25.215765
2   stale_90d_plus   1351     13.123672


/tmp/ipykernel_4505/161891197.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table_1 = signal1.groupby('staleness_bucket').agg(


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
signal2 = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        gsc_avg_position,
        gsc_clicks,
        gsc_impressions,
        CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END as ctr
    FROM '{fact_march_path}'
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
""").df()

signal2['position_bucket'] = pd.cut(
    signal2['gsc_avg_position'],
    bins=[0, 3, 10, 20, 1000],
    labels=['pos_1_3', 'pos_4_10', 'pos_11_20', 'pos_20_plus']
)

bucket_table_2 = signal2.groupby('position_bucket').agg(
    n=('content_hash_id', 'count'),
    avg_ctr=('ctr', 'mean')
).reset_index()

print(bucket_table_2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_4505/611237162.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table_2 = signal2.groupby('position_bucket').agg(


  position_bucket        n   avg_ctr
0         pos_1_3   564173  0.004918
1        pos_4_10  1456122  0.003473
2       pos_11_20   519223  0.002770
3     pos_20_plus   908354  0.001289


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
content_perf = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_avg_position) as avg_position,
        SUM(gsc_clicks) as total_clicks,
        SUM(gsc_impressions) as total_impressions
    FROM '{fact_march_path}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

content_perf['ctr'] = content_perf['total_clicks'] / content_perf['total_impressions'].replace(0, pd.NA)
content_perf['position_bucket'] = pd.cut(
    content_perf['avg_position'],
    bins=[0, 3, 10, 20, 1000],
    labels=['pos_1_3', 'pos_4_10', 'pos_11_20', 'pos_20_plus']
)

expected_ctr = content_perf.groupby('position_bucket')['ctr'].transform('median')
content_perf['ctr_gap'] = expected_ctr - content_perf['ctr']

content_perf['score'] = content_perf['ctr_gap'] * content_perf['total_impressions']
content_perf['reason_code'] = 'CTR_BELOW_EXPECTED_FOR_POSITION'
content_perf['action'] = 'CTR_FIX'

queue = content_perf.sort_values('score', ascending=False).reset_index(drop=True)
queue.head(10)

/tmp/ipykernel_4505/1802454280.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  expected_ctr = content_perf.groupby('position_bucket')['ctr'].transform('median')


,content_hash_id,client_hash_id,avg_position,total_clicks,total_impressions,ctr,position_bucket,ctr_gap,score,reason_code,action
0,content_4d0dafdb2450a480,client_20259bd6705d81d4,42.466667,0.0,15.0,0.0,pos_20_plus,0.0,0.0,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,4.909314,0.0,34.0,0.0,pos_4_10,0.0,0.0,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,4.074107,0.0,77.0,0.0,pos_4_10,0.0,0.0,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,5.177774,0.0,329.0,0.0,pos_4_10,0.0,0.0,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
4,content_2d8e52b436a736c8,client_20259bd6705d81d4,7.333333,0.0,3.0,0.0,pos_4_10,0.0,0.0,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
5,content_06c3083a26f301af,client_2b4306c3ed003f01,9.000000,0.0,1.0,0.0,pos_4_10,0.0,0.0,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
6,content_4e5fa14b358fbca8,client_08a6a72ff48e62c0,45.000000,0.0,1.0,0.0,pos_20_plus,0.0,0.0,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
7,content_9cfdbb5b11e317b1,client_08a6a72ff48e62c0,5.666667,0.0,3.0,0.0,pos_4_10,0.0,0.0,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
8,content_4ce089238b13399c,client_08a6a72ff48e62c0,31.333333,0.0,3.0,0.0,pos_20_plus,0.0,0.0,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
9,content_2003f88672099999,client_08a6a72ff48e62c0,78.000000,0.0,2.0,0.0,pos_20_plus,0.0,0.0,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.